In [ ]:
#Simple Cap geometry visualisation and cap info presentation
import pyvista as pv

# File path to the VTP file
file_path = r"Your Path\bct.vtp"

# Load the VTP file
surface = pv.read(file_path)

# Get the number of elements (cells) and vertices (points) in the surface
num_elements = surface.n_cells
num_vertices = surface.n_points

# Print the number of elements and vertices
print(f"The surface is composed of {num_elements} elements.")
print(f"The surface is composed of {num_vertices} vertices.")

# Print the coordinates of the first few vertices
vertices = surface.points
print("First few vertex coordinates:")
print(vertices[:5])

# Visualize the surface with elements highlighted and coordinate system
plotter = pv.Plotter()
plotter.add_mesh(surface, color="skyblue", show_edges=True)  # Elements highlighted
plotter.add_axes(interactive=True)  # Add an interactive coordinate system
plotter.add_title("Surface with Elements Highlighted and Coordinate System")
plotter.show()

# Visualize the surface with vertices highlighted and coordinate system
plotter = pv.Plotter()
plotter.add_mesh(surface, color="skyblue", show_edges=False)  # No edges for clean view
plotter.add_points(vertices, color="red", point_size=10)  # Vertices highlighted as red points
plotter.add_axes(interactive=True)  # Add an interactive coordinate system
plotter.add_title("Surface with Vertices Highlighted and Coordinate System")
plotter.show()

In [ ]:
#Coordinates save in txt
import pyvista as pv
import numpy as np
import os

# File paths
vtp_file_path = r"Your Path\bct.vtp"
output_dir = r"Your Path\Support_files"
output_file_path = os.path.join(output_dir, "1_cap_coor.txt")

# Load the VTP file
surface = pv.read(vtp_file_path)

# Extract vertex coordinates
vertices = surface.points

# Create an array with shape (num_points, 3) for x, y, z coordinates
cap_coor_array = np.array(vertices)  # Shape (num_points, 3)

# Write coordinates to the file in tab-separated format
with open(output_file_path, "w") as file:
    file.write("X\tY\tZ\tU\tV\tW\n")  # Header
    for vertex in vertices:
        file.write(f"{vertex[0]:.6f}\t{vertex[1]:.6f}\t{vertex[2]:.6f}\t0.000000\t0.000000\t0.000000\n")

# Print some sample elements from the array
print("Sample coordinates from cap_coor_array:")
print(cap_coor_array[:5])  # Print first 5 rows for verification

print(f"Coordinates successfully saved to {output_file_path}")

In [ ]:
#Coordinate system change
import pyvista as pv
import numpy as np
import os

# File paths
vtp_file_path = r"Your Path\bct.vtp"
output_dir = r"Your Path\Support_files"
output_file_path = os.path.join(output_dir, "2_cap_coor_transformed.txt")

# Load the VTP file
surface = pv.read(vtp_file_path)

# Extract vertex coordinates
vertices = surface.points

# Define the reference point for coordinate transformation
reference_point = np.array([9.12, -19.47, 19.46])

# Subtract the reference point from all vertices to transform the coordinate system
cap_coor_array = vertices - reference_point  # Shape (num_points, 3)

# Write transformed coordinates to the file in tab-separated format
with open(output_file_path, "w") as file:
    file.write("X\tY\tZ\tU\tV\tW\n")  # Header
    for vertex in cap_coor_array:
        file.write(f"{vertex[0]:.6f}\t{vertex[1]:.6f}\t{vertex[2]:.6f}\t0.000000\t0.000000\t0.000000\n")

# Print some sample elements from the transformed array
print("Sample transformed coordinates from cap_coor_array:")
print(cap_coor_array[:5])  # Print first 5 rows for verification

print(f"Transformed coordinates successfully saved to {output_file_path}")

In [ ]:
#Coordinate system transformation
import numpy as np
import os

# File paths
input_file_path = r"Your Path\Support_files\2_cap_coor_transformed.txt"
output_file_path = r"Your Path\Support_files\3_cap_coor_mri.txt"

# Load the transformed coordinates from the file (skip the header)
data = np.loadtxt(input_file_path, skiprows=1)  # Shape (num_points, 6), columns: X_sim Y_sim Z_sim U V W

# Extract simulation coordinates (X_sim, Y_sim, Z_sim)
x_sim, y_sim, z_sim = data[:, 0], data[:, 1], data[:, 2]

# Apply the transformation
x_mri = y_sim
y_mri = -x_sim
z_mri = -z_sim

# Create the transformed array with U, V, W still set to zero
cap_coor_mri_array = np.column_stack((x_mri, y_mri, z_mri, np.zeros_like(x_mri), np.zeros_like(y_mri), np.zeros_like(z_mri)))

# Write the MRI-transformed coordinates to a new file
with open(output_file_path, "w") as file:
    file.write("X_MRI\tY_MRI\tZ_MRI\tU\tV\tW\n")  # Header
    np.savetxt(file, cap_coor_mri_array, fmt="%.6f", delimiter="\t")

# Print some sample elements from the transformed array
print("Sample MRI-transformed coordinates:")
print(cap_coor_mri_array[:5])  # Print first 5 rows for verification

print(f"MRI-transformed coordinates successfully saved to {output_file_path}")

In [ ]:
#Scaling of the coordinates to match the MRI data
import numpy as np

# File paths
input_file_path = r"Your Path\Support_files\3_cap_coor_mri.txt"
output_file_path = r"Your Path\Support_files\4_cap_coor_mri_scaled.txt"

# Load the MRI-transformed coordinates from the file (skip the header)
data = np.loadtxt(input_file_path, skiprows=1)  # Shape (num_points, 6), columns: X_MRI Y_MRI Z_MRI U V W

# Extract MRI coordinates (X_MRI, Y_MRI, Z_MRI)
x_mri, y_mri, z_mri = data[:, 0], data[:, 1], data[:, 2]

# 1. Find the current min/max of x_mri and y_mri
x_mri_min, x_mri_max = x_mri.min(), x_mri.max()
y_mri_min, y_mri_max = y_mri.min(), y_mri.max()

# 2. Compute new scaling factors to achieve desired range
desired_xmin, desired_xmax = 41, 62
desired_ymin, desired_ymax = 29, 41

scale_x = (desired_xmax - desired_xmin) / (x_mri_max - x_mri_min)
scale_y = (desired_ymax - desired_ymin) / (y_mri_max - y_mri_min)

# Keep z-scaling the same
scale_z = 191 / 29.85  # Z-axis scaling factor

# 3. Apply scaling
x_scaled = (x_mri - x_mri_min) * scale_x + desired_xmin
y_scaled = (y_mri - y_mri_min) * scale_y + desired_ymin
z_scaled = z_mri * scale_z  # Keep Z scaling as before

# Create the scaled array with U, V, W still set to zero
cap_coor_mri_scaled_array = np.column_stack((x_scaled, y_scaled, z_scaled, np.zeros_like(x_scaled), np.zeros_like(y_scaled), np.zeros_like(z_scaled)))

# Write the scaled MRI coordinates to a new file
with open(output_file_path, "w") as file:
    file.write("X\tY\tZ\tU\tV\tW\n")  # Header
    np.savetxt(file, cap_coor_mri_scaled_array, fmt="%.6f", delimiter="\t")

# Print some sample elements from the scaled array
print("Sample scaled MRI coordinates:")
print(cap_coor_mri_scaled_array[:5])  # Print first 5 rows for verification

# Print min and max values for verification
print(f"X_SCALED: min = {x_scaled.min():.3f}, max = {x_scaled.max():.3f}")
print(f"Y_SCALED: min = {y_scaled.min():.3f}, max = {y_scaled.max():.3f}")
print(f"Z_SCALED: min = {z_scaled.min():.3f}, max = {z_scaled.max():.3f}")

print(f"Scaled MRI coordinates successfully saved to {output_file_path}")

In [ ]:
#Save it for later (Official)
import numpy as np

# File paths
input_file_path = r"Your Path\Support files\cap_coor_mri.txt"
output_file_path = r"Your Path\Support files\cap_coor_mri_scaled.txt"

# Load the MRI-transformed coordinates from the file (skip the header)
data = np.loadtxt(input_file_path, skiprows=1)  # Shape (num_points, 6), columns: X_MRI Y_MRI Z_MRI U V W

# Extract MRI coordinates (X_MRI, Y_MRI, Z_MRI)
x_mri, y_mri, z_mri = data[:, 0], data[:, 1], data[:, 2]

# Scaling factors based on provided MRI domain dimensions
scale_x = 191 / 28.66  # X-axis scaling factor 
scale_y = 71 / 17.07   # Y-axis scaling factor 
scale_z = 191 / 29.85  # Z-axis scaling factor

# Apply scaling transformation
x_scaled = x_mri * scale_x
y_scaled = y_mri * scale_y
z_scaled = z_mri * scale_z

# Create the scaled array with U, V, W still set to zero
cap_coor_mri_scaled_array = np.column_stack((x_scaled, y_scaled, z_scaled, np.zeros_like(x_scaled), np.zeros_like(y_scaled), np.zeros_like(z_scaled)))

# Write the scaled MRI coordinates to a new file
with open(output_file_path, "w") as file:
    file.write("X_SCALED\tY_SCALED\tZ_SCALED\tU\tV\tW\n")  # Header
    np.savetxt(file, cap_coor_mri_scaled_array, fmt="%.6f", delimiter="\t")

# Print some sample elements from the scaled array
print("Sample scaled MRI coordinates:")
print(cap_coor_mri_scaled_array[:5])  # Print first 5 rows for verification

# Print min and max values for X_SCALED, Y_SCALED, Z_SCALED
print(f"X_SCALED: min = {x_scaled.min():.3f}, max = {x_scaled.max():.3f}")
print(f"Y_SCALED: min = {y_scaled.min():.3f}, max = {y_scaled.max():.3f}")
print(f"Z_SCALED: min = {z_scaled.min():.3f}, max = {z_scaled.max():.3f}")

print(f"Scaled MRI coordinates successfully saved to {output_file_path}")
